# TRL SFTTrainer supervised baseline

**Objective:** prompt-completion examples; ensure completion-only masking, save eval loss; compare trusted heldout before-after.

This is a guided training starter. Training is deliberately **not executed** until you complete the numbered checks below. Prerequisite: [GRPO mechanics lab](01_grpo_clipping_diagnostics.ipynb). Save the checkpoint and metric provenance for the next lab.


# Learning companion: 02 sfttrainer baseline

**Where this fits:** This lab follows the connected APT curriculum and trains one research skill rather than just producing code. Its student exercises below remain unfinished on purpose. Read the explanation and worked example before implementing anything.

### Why this problem matters

SFT learns from supervised answers rather than reward comparisons.

### Concepts and definitions

A prompt contains a task, and a completion contains the reference response. For a causal LM, the objective maximizes completion-token likelihood conditioned on the prompt.

**Core mathematical object**

$$\mathcal{L}_{SFT}=-\frac{1}{|\mathcal{M}|}\sum_{t\in\mathcal{M}}\log p_\theta(y_t\mid x,y_{<t})$$

Interpret every quantity with its denominator, conditioning context, measurement source and train/eval boundary; a formula is not evidence until those details are fixed.

### Worked example (separate from your exercise)

If the prompt has 8 tokens and completion 4, completion-only supervision should contribute 4 positions to the loss (subject to tokenizer/chat-template boundaries), not all 12. Ignoring prompt positions is about *loss masking*, not dropping the prompt from attention.

**Reason through the example:** Which conclusion is justified directly by the given observations? Which alternative explanation remains possible? What extra measurement would discriminate between them? This is the habit that the APT project is trying to build.

### Guided code orientation and independent lab

Before writing code, identify the input record fields, the output artifact and the invariants the transformation must preserve. If the supplied scaffold imports TRL or another framework, inspect the runtime version and print the relevant API signature before running training.

**Your implementation assignment:** Inspect the tokenizer chat template and response boundary; calculate the number and fraction of supervised tokens; then compare train/eval loss before interpreting accuracy.

Use the existing notebook TODOs as your working area. You own the core transformation, metric arithmetic, split logic and tests. The notebook provides setup and rendering scaffolds rather than filling these steps for you.

### Diagnostic graph assignment

Loss versus optimizer step and supervised-token fraction; a small training loss alone cannot establish held-out coding success.

For each graph specify: x/y meaning and units, denominator, aggregation window, missing values, uncertainty, source and checkpoint. Calculate the plotting arrays yourself, then use the existing publication-quality render helper or adapt its labels and export settings.

### Interpretation and transfer

Write (1) measured observation, (2) plausible competing hypotheses, (3) confounder or leakage check, (4) discriminating next experiment, and (5) saved artifact consumed by the next lab. Do not interpret training loss or synthetic examples as trusted task success.

In [ ]:
# Colab GPU recommended. Validate TRL version and tokenizer/model requirements.
%pip -q install -U "trl" "transformers" "datasets" "accelerate" "matplotlib"

import json, inspect
import trl
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
print("TRL", trl.__version__)
print("Trainer constructor:", inspect.signature(SFTTrainer))
print("Config constructor:", inspect.signature(SFTConfig))

# TODO(student): choose a small compatible causal LM/checkpoint based on actual GPU memory.
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"


In [ ]:
# Toy learning objective; replace with an independently reviewed coding demonstration dataset.
examples = [
    {"prompt": "Write a Python function that returns the maximum of two integers.", "completion": "def max_two(a, b):\n    return a if a >= b else b"},
    {"prompt": "Write a Python function that squares an integer.", "completion": "def square(x):\n    return x * x"},
]
dataset = Dataset.from_list(examples)
config = SFTConfig(
    output_dir="artifacts/sft-baseline",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    logging_steps=1,
    max_length=256,
    report_to="none",
)
trainer = SFTTrainer(model=MODEL_ID, args=config, train_dataset=dataset)
# TODO(student): inspect tokenized labels, completion masking and train/eval split BEFORE training.
# TODO(student): add trusted held-out evaluator; compare before/after checkpoints.
# Uncomment ONLY after data/mask checks:
# trainer.train()
# trainer.save_model("artifacts/sft-baseline/final")


## Diagnostics and plotting assignment
1. Define the numerical quantities and denominators. 2. Extract training/eval measurements. 3. Compare the appropriate checkpoints and controlled baselines. 4. Interpret discrepancies and design next discriminating probe.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def plot_metric(x, y, title, ylabel, path):
    """Rendering ONLY. Student must implement x/y calculation from real trainer logs."""
    fig, ax = plt.subplots(figsize=(7.2, 4.5), layout="constrained")
    ax.plot(x, y, marker=".", linewidth=1.8)
    ax.set(title=title, xlabel="optimizer step", ylabel=ylabel)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.25)
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()

# TODO(student): extract metric points with provenance from trainer.state.log_history.
# TODO(student): choose correct step axis, smoothing policy, and diagnostic comparisons.
# plot_metric(steps, values, "Training diagnostic", "Value", "artifacts/diagnostic.png")


## Exit criteria
- Record model revision, TRL version, tokenizer, seed and GPU.
- Confirm split integrity, verifier correctness, and no label/answer leakage.
- Plot at least three diagnostics with student-computed axes and quantities.
- Save an experiment report with measured outcomes versus hypotheses.
- Do not claim Harbor multi-turn GRPO integration from this notebook.
